**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Intro to RAPIDS

> ⚠️ **Draft — requires an NVIDIA GPU; code cells not executed here.** This notebook was authored on a machine without CUDA, so unlike most workshops its cells ship without saved outputs. Run it on a CUDA machine (or [Colab](https://colab.research.google.com/) with a GPU runtime, where RAPIDS installs in ~2 min) and an instructor should verify each cell before teaching. Remove this banner after that pass.

The [GPU workshop](./Intro_GPU.ipynb) accelerated *arrays* with CuPy. RAPIDS extends the same move up the stack: **cuDF** is pandas on the GPU, **cuML** is scikit-learn on the GPU — same APIs, different silicon. The workshop's real subject is *benchmarking honestly*: when does the GPU actually win, and when are you just paying the PCIe toll?

## 0. Introduction

RAPIDS exists because dataframe work — parse, filter, join, group — is data-parallel too. The same caveat from [Intro to GPU Systems §3.3](./Intro_GPU.ipynb) governs everything: **transfers dominate**. The wins come from keeping the whole pipeline on-device.

## 1. Pre-requisites

- [Intro to Python](../Intro_Func_Prog/Intro_Python/Intro_Python.ipynb) (NumPy); pandas basics helpful.
- [Intro to GPU Systems](./Intro_GPU.ipynb) — the host/device mental model is assumed.
- Install (conda, CUDA 12.x): `conda create -n rapids -c rapidsai -c conda-forge -c nvidia rapids=24.06 python=3.11 cuda-version=12.2`
- On Colab: GPU runtime, then follow the [RAPIDS Colab install](https://rapids.ai/) cell.

---
### 🕐 Session 1 of 2 — *cuDF: Dataframes on the Device* (~35 min)
**Goal:** port a pandas pipeline to cuDF; benchmark honestly, transfers included.
**Builds on:** [Intro to GPU Systems](./Intro_GPU.ipynb). &nbsp; **Feeds into:** Session 2 (cuML).

---

## 2. cuDF

💡 **Intuition.** cuDF stores each column as a contiguous GPU array (Apache Arrow layout) — a groupby becomes thousands of threads binning rows in parallel. The API is deliberately pandas-shaped so the *port* is cheap; whether the *run* is faster depends entirely on data size and transfer count, which is why we benchmark before believing.

In [ ]:
import numpy as np
import pandas as pd
import cudf                       # GPU dataframe — pandas-shaped on purpose
import time

# Synthetic sensor log, in the spirit of the Databases workshop schema
N = 10_000_000
rng = np.random.default_rng(0)
pdf = pd.DataFrame({
    "sensor_id": rng.integers(0, 200, N),
    "value":     rng.standard_normal(N),
    "quality":   rng.integers(0, 4, N),
})
gdf = cudf.from_pandas(pdf)        # ← host→device transfer: this line COSTS
print(type(gdf), len(gdf))

In [ ]:
def pipeline(df):
    good = df[df.quality >= 2]
    stats = good.groupby("sensor_id").value.agg(["mean", "std", "count"])
    return stats.sort_index()

tic = time.perf_counter(); cpu_out = pipeline(pdf);  t_cpu = time.perf_counter() - tic
tic = time.perf_counter(); gpu_out = pipeline(gdf);  t_gpu = time.perf_counter() - tic
# fair check: results must MATCH (bring GPU result back for comparison)
ok = np.allclose(cpu_out["mean"].values, gpu_out["mean"].to_pandas().values, atol=1e-6)
print(f"pandas: {t_cpu:.3f} s   cuDF: {t_gpu:.3f} s   results match: {ok}")

### 2.1. The Honest Benchmark

The timing above *flatters* the GPU: `gdf` was already resident. An end-to-end comparison must charge the transfer — and small data flips the verdict:

In [ ]:
for n in [10_000, 1_000_000, 10_000_000]:
    small = pdf.iloc[:n]
    tic = time.perf_counter()
    _ = pipeline(cudf.from_pandas(small))      # transfer + compute, all charged
    t_gpu_e2e = time.perf_counter() - tic
    tic = time.perf_counter()
    _ = pipeline(small)
    t_cpu2 = time.perf_counter() - tic
    print(f"n = {n:>10,}:  pandas {t_cpu2:.4f} s   cuDF end-to-end {t_gpu_e2e:.4f} s")
# Expect: pandas wins small n; cuDF wins large n. Find YOUR crossover.

Moral (same as [GPU workshop §3.3](./Intro_GPU.ipynb)): the GPU wins when data is big and **stays on the device across the whole pipeline** — load with `cudf.read_parquet` directly on the GPU rather than round-tripping through pandas.

---
### 🕐 Session 2 of 2 — *cuML: Machine Learning on the Device* (~35 min)
**Goal:** GPU-accelerate scikit-learn-style fits; know the decision rule for when RAPIDS earns its keep.
**Builds on:** Session 1.

---

## 3. cuML

In [ ]:
from cuml.cluster import KMeans as cuKMeans
from sklearn.cluster import KMeans as skKMeans
import cupy as cp

X_host = rng.standard_normal((2_000_000, 8)).astype(np.float32)
X_dev  = cp.asarray(X_host)                      # explicit, charged transfer

tic = time.perf_counter()
sk = skKMeans(n_clusters=8, n_init=1, random_state=0).fit(X_host)
t_sk = time.perf_counter() - tic

tic = time.perf_counter()
cu = cuKMeans(n_clusters=8, n_init=1, random_state=0).fit(X_dev)
t_cu = time.perf_counter() - tic

print(f"scikit-learn: {t_sk:.2f} s   cuML: {t_cu:.2f} s")
print(f"inertia agreement (relative): {abs(sk.inertia_ - float(cu.inertia_)) / sk.inertia_:.2e}")

💡 **Intuition.** K-means is distance computations in a loop — exactly the dense, regular arithmetic GPUs devour. The pattern generalizes: cuML shines on **iterative, matrix-heavy** fits (KMeans, PCA, linear/logistic, UMAP, random forests) and adds little when the model is tiny or the fit is I/O-bound.

**Decision rule to teach:** (1) data ≥ millions of rows, (2) pipeline stays on-device end to end, (3) the algorithm is compute-bound → RAPIDS. Otherwise pandas/sklearn are simpler and often faster. Measure, don't assume — you now know how to measure honestly.

## 4. Conclusion

cuDF and cuML move the [GPU workshop's](./Intro_GPU.ipynb) lesson up the stack: same APIs you know, massive wins **iff** data is large and transfers are amortized. The benchmark discipline — charge the transfer, verify the outputs match, find the crossover — is the transferable skill.

---
## Where next

- [Intro to Databases](../Intro_Host_Prog/Intro_Databases/Intro_Databases.ipynb) — "arrays in files, metadata in SQL" pairs with `cudf.read_parquet` beautifully.
- [Scaling Neural Networks](../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb) — the same bottleneck-hunting mindset applied to training.
- [Intro to GPU Systems](./Intro_GPU.ipynb) — the memory model underneath every timing above.